In [23]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.backends.backend_pdf import PdfPages
from scipy.spatial.distance import cosine, pdist

In [24]:
CT_COL     = "TCRClonotype"
SAMPLE_COL = "Sample_Origin"
LY49C_COL  = "Ly-49C-Ly-49I-Klra3-Klra9-AMM2139-pAbO"

min_size = 5
plots_per_page = 16
thresholds = np.linspace(1.0, 0.85, 16)
default_q = 0.925

DEBUG_PROBS = False

In [25]:
markers_b10br = [
    "RiO-Allo:H-2Kb-ATLVFHNL-pAbO",
    "RiO-Allo:H-2Kb-EEEPVKKI-pAbO",
    "RiO-Allo:H-2Kb-HIYEFPQL-pAbO",
    "RiO-Allo:H-2Kb-INFDFPKL-pAbO",
    "RiO-Allo:H-2Kb-RAYLFNSV-pAbO",
    "RiO-Allo:H-2Kb-RTYTYEKL-pAbO",
    "RiO-Allo:H-2Kb-SNYLFTKL-pAbO",
    "RiO-Allo:H-2Kb-SSYTFPKM-pAbO",
    "RiO-Allo:H-2Kb-SVYVYKVL-pAbO",
    "RiO-Allo:H-2Kb-VAFDFTKV-pAbO",
    "RiO-Allo:H-2Kb-VGPRYTNL-pAbO",
    "RiO-Allo:H-2Kb-VIVRFLTV-pAbO",
    "RiO-Allo:H-2Kb-VSFTYRYL-pAbO",
]

peptides_b10br = [
    "ATLVFHNL","EEEPVKKI","HIYEFPQL","INFDFPKL","RAYLFNSV","RTYTYEKL",
    "SNYLFTKL","SSYTFPKM","SVYVYKVL","VAFDFTKV","VGPRYTNL","VIVRFLTV","VSFTYRYL"
]

markers_balbc = [
    "RiO-Allo:H-2Kb-ATLVFHNL-pAbO",
    "RiO-Allo:H-2Kb-HIYEFPQL-pAbO",
    "RiO-Allo:H-2Kb-INFDFPKL-pAbO",
    "RiO-Allo:H-2Kb-RAYLFNSV-pAbO",
    "RiO-Allo:H-2Kb-RTYTYEKL-pAbO",
    "RiO-Allo:H-2Kb-SNYLFTKL-pAbO",
    "RiO-Allo:H-2Kb-SSYTFPKM-pAbO",
    "RiO-Allo:H-2Kb-SVYVYKVL-pAbO",
    "RiO-Allo:H-2Kb-VAFDFTKV-pAbO",
    "RiO-Allo:H-2Kb-VGPRYTNL-pAbO",
    "RiO-Allo:H-2Kb-VIVRFLTV-pAbO",
    "RiO-Allo:H-2Kb-VSFTYRYL-pAbO",
    "RiO-H-2:H-2Kd-SYFPEITHI-ADEX5099-pAbO"
]

peptides_balbc = [
    "ATLVFHNL","HIYEFPQL","INFDFPKL","RAYLFNSV","RTYTYEKL","SNYLFTKL",
    "SSYTFPKM","SVYVYKVL","VAFDFTKV","VGPRYTNL","VIVRFLTV","VSFTYRYL","SYFPEITHI"
]

In [26]:
def row_normalise(
    X: np.ndarray,
    eps: float = 0.0,
    debug: bool = False,
    tag: str = "P",
) -> np.ndarray:
    """
    Row-normalise raw peptide intensities/counts -> probabilities per cell.

    If a row sums to 0 (all peptides 0), we return an all-zero probability row.
    Those contribute 0 entropy under our convention (see renyi_entropy).
    """
    X = np.asarray(X, dtype=float)
    rs = X.sum(axis=1, keepdims=True)

    zero = (rs.squeeze() <= 0)
    if debug:
        frac = 100.0 * float(np.mean(zero)) if X.shape[0] else 0.0
        print(f"[DEBUG] {tag}: {frac:.2f}% cells have zero RiO mass (all peptides 0).")

    rs_safe = rs.copy()
    rs_safe[rs_safe <= 0] = 1.0
    P = X / rs_safe

    if eps > 0:
        P = np.maximum(P, eps)
        P = P / P.sum(axis=1, keepdims=True)

    return P

In [27]:
def renyi_entropy(P: np.ndarray, alpha: float, axis: int = -1) -> np.ndarray:
    """
    Renyi entropy in bits for probability vectors P.

    IMPORTANT: Handles all-zero rows safely (returns 0 for those rows).
      alpha=1 -> Shannon
      alpha=0 -> Hartley (log2 support size)
    """
    P = np.asarray(P, dtype=float)
    s = P.sum(axis=axis)
    zero_mask = (s == 0)

    def apply_zero_mask(H):
        if np.isscalar(H):
            return 0.0 if zero_mask else H
        H = np.asarray(H, dtype=float)
        H[zero_mask] = 0.0
        return H

    if alpha == 1.0:
        with np.errstate(divide="ignore", invalid="ignore"):
            logP = np.where(P > 0, np.log2(P), 0.0)
        H = -(P * logP).sum(axis=axis)
        return apply_zero_mask(H)

    if alpha == 0.0:
        k = np.sum(P > 0, axis=axis)
        k = np.maximum(k, 1)
        H = np.log2(k)
        return apply_zero_mask(H)

    S = np.sum(np.power(np.maximum(P, 0.0), alpha), axis=axis)
    S = np.maximum(S, 1e-300)
    H = (1.0 / (1.0 - alpha)) * np.log2(S)
    return apply_zero_mask(H)

In [28]:
def mean_pairwise_cosine_similarity(P: np.ndarray) -> float:
    """
    P: row-normalised per cell (n_cells x n_peptides).
    Returns mean pairwise cosine similarity, or NaN if <2 cells.
    """
    if P.shape[0] < 2:
        return np.nan
    return float(np.mean(1.0 - pdist(P, metric="cosine")))

In [29]:
def compute_clonotype_summaries(
    df: pd.DataFrame,
    ct_col: str,
    markers: list[str],
    min_size: int = 5,
    entropy_alpha: float = 1.0,
    debug_probs: bool = False,
) -> dict:
    """
    Returns dict ct -> summary:
      - mean_pattern: mean of per-cell normalised vectors
      - mean_entropy: mean Renyi entropy (alpha) across cells
      - mean_coherence: mean pairwise cosine similarity across cells
      - n_cells: number of cells
    """
    vc = df[ct_col].value_counts()
    keep = vc[vc >= min_size].index
    df2 = df[df[ct_col].isin(keep)]
    out = {}

    for ct, g in df2.groupby(ct_col):
        X = g[markers].to_numpy(dtype=float, copy=False)
        P = row_normalise(X, debug=debug_probs, tag=f"P_all[{ct}]")

        ent = float(np.mean(renyi_entropy(P, alpha=float(entropy_alpha), axis=1)))
        coh = mean_pairwise_cosine_similarity(P)

        out[ct] = {
            "mean_pattern": P.mean(axis=0),
            "mean_entropy": ent,
            "mean_coherence": coh,
            "n_cells": int(P.shape[0]),
        }
    return out

In [30]:
def apply_ly49c_filter_one_sample(df_s: pd.DataFrame, ly49c_col: str, q: float) -> tuple[pd.DataFrame, float]:
    """
    Within ONE sample: keep Ly49C <= quantile(q).
    Returns (filtered_df, Ts).
    """
    if q >= 1.0:
        return df_s.copy(), float(df_s[ly49c_col].max())
    Ts = float(df_s[ly49c_col].quantile(q))
    return df_s[df_s[ly49c_col] <= Ts].copy(), Ts

In [31]:
def pattern_distance(p1: np.ndarray, p2: np.ndarray, metric: str) -> float:
    if metric == "cosine":
        if np.allclose(p1, 0) or np.allclose(p2, 0):
            return 1.0
        return float(cosine(p1, p2))
    if metric == "l1":
        return float(np.sum(np.abs(p1 - p2)))
    raise ValueError(f"Unknown metric: {metric}")

In [32]:
def ly49c_sweep_metrics_one_sample(
    df_s: pd.DataFrame,
    ct_col: str,
    ly49c_col: str,
    markers: list[str],
    thresholds: np.ndarray,
    min_size: int = 5,
    entropy_alpha: float = 1.0,
    debug_probs: bool = False,
) -> pd.DataFrame:
    """
    Computes sweep metrics for ONE sample (entropy order fixed).

    Outputs per q:
      - retention
      - median cosine dist between mean clonotype patterns (baseline vs filtered)
      - median L1 dist
      - mean entropy change (filtered - baseline) across common clonotypes
      - mean coherence change (filtered - baseline)
    """
    baseline = compute_clonotype_summaries(
        df_s, ct_col, markers, min_size=min_size, entropy_alpha=entropy_alpha, debug_probs=debug_probs
    )
    n0 = len(df_s)

    rows = []
    for q in thresholds:
        df_f, Ts = apply_ly49c_filter_one_sample(df_s, ly49c_col, float(q))
        filtered = compute_clonotype_summaries(
            df_f, ct_col, markers, min_size=min_size, entropy_alpha=entropy_alpha, debug_probs=debug_probs
        )

        common = set(baseline) & set(filtered)
        if common:
            cos_d = [pattern_distance(baseline[c]["mean_pattern"], filtered[c]["mean_pattern"], "cosine") for c in common]
            l1_d  = [pattern_distance(baseline[c]["mean_pattern"], filtered[c]["mean_pattern"], "l1") for c in common]
            ent_d = [filtered[c]["mean_entropy"] - baseline[c]["mean_entropy"] for c in common]
            coh_d = [filtered[c]["mean_coherence"] - baseline[c]["mean_coherence"] for c in common]

            row = {
                "percentile": float(q),
                "Ts_ly49c": float(Ts),
                "n_cells": int(len(df_f)),
                "pct_cells_retained": 100.0 * len(df_f) / n0 if n0 else np.nan,
                "n_clonotypes_baseline_ge5": int(len(baseline)),
                "n_clonotypes_filtered_ge5": int(len(filtered)),
                "n_common_clonotypes_ge5": int(len(common)),
                "median_cosine_dist": float(np.median(cos_d)),
                "median_l1_dist": float(np.median(l1_d)),
                "mean_entropy_change": float(np.mean(ent_d)),
                "mean_coherence_change": float(np.nanmean(coh_d)),
            }
        else:
            row = {
                "percentile": float(q),
                "Ts_ly49c": float(Ts),
                "n_cells": int(len(df_f)),
                "pct_cells_retained": 100.0 * len(df_f) / n0 if n0 else np.nan,
                "n_clonotypes_baseline_ge5": int(len(baseline)),
                "n_clonotypes_filtered_ge5": int(len(filtered)),
                "n_common_clonotypes_ge5": 0,
                "median_cosine_dist": np.nan,
                "median_l1_dist": np.nan,
                "mean_entropy_change": np.nan,
                "mean_coherence_change": np.nan,
            }

        rows.append(row)

    return pd.DataFrame(rows)

In [33]:
def ly49c_sweep_entropy_orders_one_sample(
    df_s: pd.DataFrame,
    sample_name: str,
    ct_col: str,
    ly49c_col: str,
    markers: list[str],
    thresholds: np.ndarray,
    orders: list[float],
    min_size: int = 5,
    debug_probs: bool = False,
) -> pd.DataFrame:
    """
    Long-form table for entropy change vs q and alpha, computed over common clonotypes.

    Returns columns:
      sample, percentile, Ts_ly49c, alpha, mean_entropy_change, n_common_clonotypes_ge5
    """
    # baseline entropies per clonotype per alpha
    base_summ = {}
    vc = df_s[ct_col].value_counts()
    keep = vc[vc >= min_size].index
    df2 = df_s[df_s[ct_col].isin(keep)]

    for ct, g in df2.groupby(ct_col):
        X = g[markers].to_numpy(dtype=float, copy=False)
        P = row_normalise(X, debug=debug_probs, tag=f"P_all[{ct}]")
        base_summ[ct] = {a: float(np.mean(renyi_entropy(P, alpha=float(a), axis=1))) for a in orders}

    rows = []
    for q in thresholds:
        df_f, Ts = apply_ly49c_filter_one_sample(df_s, ly49c_col, float(q))

        vc_f = df_f[ct_col].value_counts()
        keep_f = vc_f[vc_f >= min_size].index
        df_f2 = df_f[df_f[ct_col].isin(keep_f)]

        filt_summ = {}
        for ct, g in df_f2.groupby(ct_col):
            X = g[markers].to_numpy(dtype=float, copy=False)
            P = row_normalise(X, debug=debug_probs, tag=f"P_all[{ct}]")
            filt_summ[ct] = {a: float(np.mean(renyi_entropy(P, alpha=float(a), axis=1))) for a in orders}

        common = sorted(set(base_summ) & set(filt_summ))
        for a in orders:
            if common:
                dH = [filt_summ[ct][a] - base_summ[ct][a] for ct in common]
                mean_dH = float(np.mean(dH))
                n_common = int(len(common))
            else:
                mean_dH = np.nan
                n_common = 0

            rows.append({
                "sample": sample_name,
                "percentile": float(q),
                "Ts_ly49c": float(Ts),
                "alpha": float(a),
                "mean_entropy_change": mean_dH,
                "n_common_clonotypes_ge5": n_common,
                "n_cells_after": int(len(df_f)),
            })

    return pd.DataFrame(rows)

In [34]:
def clonotype_raw_and_prop_from_raw(
    df: pd.DataFrame,
    ct_col: str,
    markers: list[str],
    min_size: int = 5,
) -> tuple[dict, dict, dict]:
    """
    For clonotypes with >=min_size cells:
      raw_sum[ct] = sum of raw peptide counts across cells
      prop[ct]    = raw_sum / raw_sum.sum()
      n_cells[ct] = number of cells in clonotype (in THIS df)
    """
    vc = df[ct_col].value_counts()
    keep = vc[vc >= min_size].index
    df2 = df[df[ct_col].isin(keep)]

    raw_sum, prop = {}, {}
    n_cells = {k: int(v) for k, v in vc[keep].to_dict().items()}

    for ct, g in df2.groupby(ct_col):
        X = g[markers].to_numpy(dtype=float, copy=False)
        s = X.sum(axis=0)
        raw_sum[ct] = s
        tot = s.sum()
        prop[ct] = s / tot if tot > 0 else np.zeros_like(s)

    return raw_sum, prop, n_cells

In [35]:
def plot_raw_and_prop_before_after_4x8_pdf(
    raw_b: dict, raw_a: dict,
    prop_b: dict, prop_a: dict,
    n_b: dict, n_a: dict,
    peptides: list[str],
    out_pdf: Path,
    title: str,
):
    xs = np.arange(len(peptides))
    w = 0.42
    per_page = 16

    clonotypes = sorted(n_b.keys(), key=lambda c: n_b[c], reverse=True)

    with PdfPages(out_pdf) as pdf:
        for start in range(0, len(clonotypes), per_page):
            chunk = clonotypes[start:start + per_page]

            fig, axes = plt.subplots(4, 8, figsize=(24, 12))
            axes = axes.flatten()

            for i, ct in enumerate(chunk):
                ax_raw = axes[2*i]
                ax_prp = axes[2*i + 1]
                has_after = ct in raw_a

                # Raw
                b = raw_b[ct]
                if has_after:
                    a = raw_a[ct]
                    ax_raw.bar(xs - w/2, b, w, alpha=0.85, label="Before")
                    ax_raw.bar(xs + w/2, a, w, alpha=0.85, label="After")
                    ax_raw.set_title(f"{ct}\nRAW n:{n_b[ct]}→{n_a.get(ct,0)}", fontsize=7)
                else:
                    ax_raw.bar(xs, b, w*1.8, alpha=0.6, label="Before (lost)")
                    ax_raw.set_title(f"{ct}\nRAW n:{n_b[ct]}→0 LOST", fontsize=7, color="darkred")

                ax_raw.set_xticks(xs)
                ax_raw.set_xticklabels(peptides, rotation=90, fontsize=6)
                ax_raw.tick_params(axis="y", labelsize=6)
                ax_raw.spines["top"].set_visible(False)
                ax_raw.spines["right"].set_visible(False)
                if i == 0:
                    ax_raw.legend(fontsize=7, loc="upper right")

                # Proportions
                b = prop_b[ct]
                if has_after:
                    a = prop_a[ct]
                    ax_prp.bar(xs - w/2, b, w, alpha=0.85, label="Before")
                    ax_prp.bar(xs + w/2, a, w, alpha=0.85, label="After")
                    ax_prp.set_title(f"{ct}\nPROP n:{n_b[ct]}→{n_a.get(ct,0)}", fontsize=7)
                else:
                    ax_prp.bar(xs, b, w*1.8, alpha=0.6, label="Before (lost)")
                    ax_prp.set_title(f"{ct}\nPROP n:{n_b[ct]}→0 LOST", fontsize=7, color="darkred")

                ax_prp.set_xticks(xs)
                ax_prp.set_xticklabels(peptides, rotation=90, fontsize=6)
                ax_prp.tick_params(axis="y", labelsize=6)
                ax_prp.spines["top"].set_visible(False)
                ax_prp.spines["right"].set_visible(False)

            for j in range(2 * len(chunk), 32):
                axes[j].axis("off")

            page = start // per_page + 1
            n_pages = (len(clonotypes) + per_page - 1) // per_page
            fig.suptitle(f"{title} (page {page}/{n_pages})", fontsize=14, y=0.995)
            fig.tight_layout(rect=[0, 0, 1, 0.97])
            pdf.savefig(fig)
            plt.close(fig)

In [36]:
def renyi_profile_all_clonotypes_ge5(
    df_s: pd.DataFrame,
    ct_col: str,
    markers: list[str],
    min_size: int = 5,
    orders: list[float] | None = None,
    debug: bool = False,
) -> pd.DataFrame:
    """
    For ALL clonotypes with >= min_size cells in ONE sample:
      compute mean Renyi entropy per clonotype for each alpha in `orders`.

    Returns long-form: clonotype, n_cells, alpha, renyi_entropy_mean
    """
    if orders is None:
        orders = [0, 1, 2, 3, 4, 5]

    vc = df_s[ct_col].value_counts()
    keep = vc[vc >= min_size].index
    df2 = df_s[df_s[ct_col].isin(keep)]

    rows = []
    for ct, g in df2.groupby(ct_col):
        X = g[markers].to_numpy(dtype=float, copy=False)
        P = row_normalise(X, debug=debug, tag=f"P_all[{ct}]")
        n_cells = int(P.shape[0])

        for a in orders:
            h = float(np.mean(renyi_entropy(P, alpha=float(a), axis=1)))
            rows.append({"clonotype": ct, "n_cells": n_cells, "alpha": float(a), "renyi_entropy_mean": h})

    return pd.DataFrame(rows)

In [37]:
def plot_renyi_spaghetti_all_clonotypes(
    df_long: pd.DataFrame,
    out_png: Path,
    title: str,
    alpha_linewidth: float = 1.7,
    alpha_opacity: float = 0.55,
):
    """
    df_long columns: clonotype, alpha, renyi_entropy_mean
    Plots one line per clonotype (spaghetti), darker & thicker.
    """
    fig, ax = plt.subplots(figsize=(8.2, 6.2))

    if df_long.empty or "clonotype" not in df_long.columns:
        ax.text(0.5, 0.5, "No clonotypes ≥ min_size", 
                ha="center", va="center", transform=ax.transAxes, fontsize=12)
        ax.set_xlabel("Renyi order α")
        ax.set_ylabel("Mean Renyi entropy (bits)")
        ax.set_title(title)
        fig.tight_layout()
        fig.savefig(out_png, dpi=170, bbox_inches="tight")
        plt.close(fig)
        return

    for ct, g in df_long.groupby("clonotype"):
        g2 = g.sort_values("alpha")
        ax.plot(
            g2["alpha"], g2["renyi_entropy_mean"],
            marker="o",
            lw=alpha_linewidth,
            alpha=alpha_opacity
        )

    ax.set_xlabel("Renyi order α")
    ax.set_ylabel("Mean Renyi entropy (bits)")
    ax.set_title(title)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    fig.tight_layout()
    fig.savefig(out_png, dpi=170, bbox_inches="tight")
    plt.close(fig)

In [38]:
def plot_entropy_sweep_overlay_alpha_by_sample_2x3(
    df_long: pd.DataFrame,
    out_png: Path,
    title: str,
    chosen_q: float | None = None,
    max_panels: int = 6,
):
    """
    Expects df_long columns: sample, percentile, alpha, mean_entropy_change
    Produces 2x3 grid of samples; within each panel overlays α curves.
    """
    samples = list(df_long["sample"].dropna().unique())[:max_panels]

    fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharey=True)
    axes = axes.flatten()

    for i in range(6):
        ax = axes[i]
        if i >= len(samples):
            ax.axis("off")
            continue

        s = samples[i]
        sub = df_long[df_long["sample"] == s].copy()
        for a, g in sub.groupby("alpha"):
            g2 = g.sort_values("percentile")
            ax.plot(g2["percentile"], g2["mean_entropy_change"], marker="o", lw=1.6, alpha=0.9, label=f"α={a:g}")

        if chosen_q is not None:
            ax.axvline(chosen_q, color="red", ls="--", alpha=0.7)

        ax.set_title(str(s))
        ax.set_xlabel("q (Ly49C percentile)")
        ax.invert_xaxis()
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.legend(frameon=False, fontsize=8, ncol=2)

    fig.suptitle(title, y=0.98)
    fig.text(0.5, 0.04, "q (Ly49C percentile, within sample)", ha="center")
    fig.text(0.04, 0.5, "Mean ΔHα (after − before) [bits]", va="center", rotation="vertical")
    fig.tight_layout(rect=[0.06, 0.06, 1, 0.95])
    fig.savefig(out_png, dpi=170, bbox_inches="tight")
    plt.close(fig)

In [39]:
def coherence_before_after_table(
    df_before: pd.DataFrame,
    df_after: pd.DataFrame,
    ct_col: str,
    markers: list[str],
    min_size: int = 5,
    debug_probs: bool = False,
) -> pd.DataFrame:
    base = compute_clonotype_summaries(
        df_before, ct_col, markers, min_size=min_size, entropy_alpha=1.0, debug_probs=debug_probs
    )
    aft = compute_clonotype_summaries(
        df_after, ct_col, markers, min_size=min_size, entropy_alpha=1.0, debug_probs=debug_probs
    )

    common = sorted(set(base) & set(aft))
    
    if not common:
        return pd.DataFrame(columns=["clonotype", "coh_before", "coh_after", "n_before", "n_after"])
    
    rows = []
    for ct in common:
        rows.append({
            "clonotype": ct,
            "coh_before": float(base[ct]["mean_coherence"]),
            "coh_after": float(aft[ct]["mean_coherence"]),
            "n_before": int(base[ct]["n_cells"]),
            "n_after": int(aft[ct]["n_cells"]),
        })
    return pd.DataFrame(rows)

In [40]:
def plot_coherence_before_after_scatter(
    df_pairs: pd.DataFrame,
    out_png: Path,
    title: str,
    mean_coherence_change: float | None = None,
):
    """
    Scatter plot of coherence before vs after filtering.
    Now includes mean coherence change annotation if provided.
    """
    fig, ax = plt.subplots(figsize=(6.5, 6.5))

    if df_pairs.empty or "coh_before" not in df_pairs.columns:
        ax.text(0.5, 0.5, "No common clonotypes ≥ min_size", 
                ha="center", va="center", transform=ax.transAxes, fontsize=12)
        ax.set_xlabel("Mean coherence (baseline)")
        ax.set_ylabel("Mean coherence (filtered)")
        ax.set_title(title)
        fig.tight_layout()
        fig.savefig(out_png, dpi=170, bbox_inches="tight")
        plt.close(fig)
        return

    x = df_pairs["coh_before"].to_numpy()
    y = df_pairs["coh_after"].to_numpy()

    ax.scatter(x, y, s=18, alpha=0.7)

    lo = np.nanmin(np.r_[x, y])
    hi = np.nanmax(np.r_[x, y])
    ax.plot([lo, hi], [lo, hi], ls="--", color="black", alpha=0.6)

    ax.set_xlabel("Mean coherence (baseline)")
    ax.set_ylabel("Mean coherence (filtered)")
    ax.set_title(title)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # Build annotation text
    annot_lines = []
    if len(df_pairs) >= 3 and np.isfinite(x).all() and np.isfinite(y).all():
        r = float(np.corrcoef(x, y)[0, 1])
        annot_lines.append(f"r = {r:.2f}")
    annot_lines.append(f"N = {len(df_pairs)}")
    
    # Add mean coherence change if provided
    if mean_coherence_change is not None and np.isfinite(mean_coherence_change):
        sign = "+" if mean_coherence_change >= 0 else ""
        annot_lines.append(f"Mean Δcoh = {sign}{mean_coherence_change:.3f}")
    
    ax.text(0.02, 0.98, "\n".join(annot_lines), transform=ax.transAxes,
            va="top", ha="left", fontsize=10)

    fig.tight_layout()
    fig.savefig(out_png, dpi=170, bbox_inches="tight")
    plt.close(fig)

In [41]:
def plot_sweep_metrics_2x3(
    sweep_df: pd.DataFrame,
    sweep_entropy_df: pd.DataFrame,
    out_png: Path,
    title: str,
    chosen_q: float | None = None,
):
    """
    2x3 panel plot showing sweep metrics:
      [0,0] Median cosine distance
      [0,1] Median L1 distance
      [0,2] Mean entropy change (α=0,1,2,3,4,5 overlaid)
      [1,0] Mean coherence change
      [1,1] Data retention (% cells retained)
      [1,2] Number of common clonotypes ≥ min_size
    """
    fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharex=True)
    
    q_vals = sweep_df["percentile"].to_numpy()
    
    # [0,0] Median cosine distance
    ax = axes[0, 0]
    ax.plot(q_vals, sweep_df["median_cosine_dist"], marker="o", lw=1.8, color="C0")
    ax.set_ylabel("Median cosine distance")
    ax.set_title("Pattern shift (cosine)")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if chosen_q is not None:
        ax.axvline(chosen_q, color="red", ls="--", alpha=0.7)
    ax.invert_xaxis()
    
    # [0,1] Median L1 distance
    ax = axes[0, 1]
    ax.plot(q_vals, sweep_df["median_l1_dist"], marker="o", lw=1.8, color="C1")
    ax.set_ylabel("Median L1 distance")
    ax.set_title("Pattern shift (L1)")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if chosen_q is not None:
        ax.axvline(chosen_q, color="red", ls="--", alpha=0.7)
    
    # [0,2] Mean entropy change (α=0..5 overlaid)
    ax = axes[0, 2]
    colors = plt.cm.viridis(np.linspace(0.1, 0.9, 6))
    for i, alpha_val in enumerate([0, 1, 2, 3, 4, 5]):
        sub = sweep_entropy_df[sweep_entropy_df["alpha"] == alpha_val].sort_values("percentile")
        if not sub.empty:
            ax.plot(sub["percentile"], sub["mean_entropy_change"], 
                   marker="o", lw=1.5, alpha=0.85, color=colors[i], label=f"α={alpha_val}")
    ax.set_ylabel("Mean ΔH (bits)")
    ax.set_title("Entropy change by α")
    ax.legend(frameon=False, fontsize=8, ncol=2, loc="best")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if chosen_q is not None:
        ax.axvline(chosen_q, color="red", ls="--", alpha=0.7)
    
    # [1,0] Mean coherence change
    ax = axes[1, 0]
    ax.plot(q_vals, sweep_df["mean_coherence_change"], marker="o", lw=1.8, color="C2")
    ax.axhline(0, color="gray", ls=":", alpha=0.5)
    ax.set_ylabel("Mean Δcoherence")
    ax.set_title("Coherence change")
    ax.set_xlabel("q (Ly49C percentile)")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if chosen_q is not None:
        ax.axvline(chosen_q, color="red", ls="--", alpha=0.7)
    
    # [1,1] Data retention
    ax = axes[1, 1]
    ax.plot(q_vals, sweep_df["pct_cells_retained"], marker="o", lw=1.8, color="C3")
    ax.set_ylabel("% cells retained")
    ax.set_title("Data retention")
    ax.set_xlabel("q (Ly49C percentile)")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if chosen_q is not None:
        ax.axvline(chosen_q, color="red", ls="--", alpha=0.7)
    
    # [1,2] Number of common clonotypes
    ax = axes[1, 2]
    ax.plot(q_vals, sweep_df["n_common_clonotypes_ge5"], marker="o", lw=1.8, color="C4")
    ax.set_ylabel("N common clonotypes ≥5")
    ax.set_title("Clonotype retention")
    ax.set_xlabel("q (Ly49C percentile)")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if chosen_q is not None:
        ax.axvline(chosen_q, color="red", ls="--", alpha=0.7)
    
    fig.suptitle(title, fontsize=12, y=0.98)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(out_png, dpi=170, bbox_inches="tight")
    plt.close(fig)

In [42]:
def run_samplewise_pipeline(
    df: pd.DataFrame,
    dataset_name: str,
    markers: list[str],
    peptides: list[str],
    output_root: Path,
    thresholds: np.ndarray,
    chosen_q: float,
    ct_col: str = CT_COL,
    sample_col: str = SAMPLE_COL,
    ly49c_col: str = LY49C_COL,
    min_size: int = 5,
    entropy_alpha_for_sweep: float = 1.0,
    renyi_orders: list[float] | None = None,
    debug_probs: bool = False,
) -> pd.DataFrame:
    """
    Creates:
      output_root/dataset_name/<sample>/
        - ly49c_sweep_metrics_alpha{alpha}.csv
        - ly49c_sweep_metrics_2x3.png  (NEW: 2x3 panel with all metrics)
        - clonotypes_RAW_and_PROP_before_after_q{chosen_q}.pdf
        - renyi_all_ge{min_size}_baseline.csv
        - renyi_all_ge{min_size}_filtered.csv
        - renyi_spaghetti_baseline.png
        - renyi_spaghetti_filtered.png
        - coherence_before_vs_after.png (+ csv) (UPDATED: now shows mean Δcoh)
      output_root/dataset_name/
        - INDEX.csv
        - ly49c_entropy_sweep_overlay_alpha0to5_2x3.png (first 6 samples)
    """
    output_root = Path(output_root)
    ds_out = output_root / dataset_name
    ds_out.mkdir(parents=True, exist_ok=True)

    orders = renyi_orders or [0, 1, 2, 3, 4, 5]

    needed = {ct_col, sample_col, ly49c_col, *markers}
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise KeyError(f"Missing required columns (first few): {missing[:10]}")

    index_rows = []
    sweep_entropy_all_samples = []

    for sample, df_s in df.groupby(sample_col):
        s_out = ds_out / str(sample)
        s_out.mkdir(parents=True, exist_ok=True)

        # --- A) Main sweep metrics (single alpha, usually Shannon)
        sweep_df = ly49c_sweep_metrics_one_sample(
            df_s=df_s,
            ct_col=ct_col,
            ly49c_col=ly49c_col,
            markers=markers,
            thresholds=thresholds,
            min_size=min_size,
            entropy_alpha=entropy_alpha_for_sweep,
            debug_probs=debug_probs,
        )
        sweep_csv = s_out / f"ly49c_sweep_metrics_alpha{entropy_alpha_for_sweep}.csv"
        sweep_df.to_csv(sweep_csv, index=False)

        # --- B) Entropy sweep for alpha 0..5 (collect for dataset-level 2x3 plot)
        sweep_ent_long = ly49c_sweep_entropy_orders_one_sample(
            df_s=df_s,
            sample_name=str(sample),
            ct_col=ct_col,
            ly49c_col=ly49c_col,
            markers=markers,
            thresholds=thresholds,
            orders=orders,
            min_size=min_size,
            debug_probs=debug_probs,
        )
        sweep_entropy_all_samples.append(sweep_ent_long)

        # --- NEW: 2x3 sweep metrics plot ---
        sweep_metrics_png = s_out / "ly49c_sweep_metrics_2x3.png"
        # Filter entropy sweep to just this sample
        sweep_ent_this_sample = sweep_ent_long[sweep_ent_long["sample"] == str(sample)]
        plot_sweep_metrics_2x3(
            sweep_df=sweep_df,
            sweep_entropy_df=sweep_ent_this_sample,
            out_png=sweep_metrics_png,
            title=f"{dataset_name} | {sample} | Ly49C sweep metrics",
            chosen_q=chosen_q,
        )

        # --- C) Apply chosen filter
        df_after, Ts = apply_ly49c_filter_one_sample(df_s, ly49c_col, chosen_q)

        # --- D) RAW+PROP before/after PDF
        raw_b, prop_b, n_b = clonotype_raw_and_prop_from_raw(df_s, ct_col, markers, min_size=min_size)
        raw_a, prop_a, n_a = clonotype_raw_and_prop_from_raw(df_after, ct_col, markers, min_size=min_size)

        pdf_out = s_out / f"clonotypes_RAW_and_PROP_before_after_q{chosen_q:.3f}.pdf"
        plot_raw_and_prop_before_after_4x8_pdf(
            raw_b, raw_a, prop_b, prop_a, n_b, n_a,
            peptides=peptides,
            out_pdf=pdf_out,
            title=f"{dataset_name} | {sample} | clonotypes≥{min_size} RAW+PROP before/after (q={chosen_q:.3f}, Ts={Ts:.2f})",
        )

        # --- E) Spaghetti entropies: ALL clonotypes >= min_size (baseline + after)
        renyi_base = renyi_profile_all_clonotypes_ge5(
            df_s=df_s, ct_col=ct_col, markers=markers, min_size=min_size, orders=orders, debug=debug_probs
        )
        renyi_after = renyi_profile_all_clonotypes_ge5(
            df_s=df_after, ct_col=ct_col, markers=markers, min_size=min_size, orders=orders, debug=debug_probs
        )

        renyi_base_csv = s_out / f"renyi_all_ge{min_size}_baseline.csv"
        renyi_after_csv = s_out / f"renyi_all_ge{min_size}_filtered_q{chosen_q:.3f}.csv"
        renyi_base.to_csv(renyi_base_csv, index=False)
        renyi_after.to_csv(renyi_after_csv, index=False)

        plot_renyi_spaghetti_all_clonotypes(
            renyi_base,
            out_png=s_out / "renyi_spaghetti_baseline.png",
            title=f"{dataset_name} | {sample} | Renyi spaghetti (ALL clonotypes≥{min_size}) | baseline",
            alpha_linewidth=1.8,
            alpha_opacity=0.6,
        )
        plot_renyi_spaghetti_all_clonotypes(
            renyi_after,
            out_png=s_out / f"renyi_spaghetti_filtered_q{chosen_q:.3f}.png",
            title=f"{dataset_name} | {sample} | Renyi spaghetti (ALL clonotypes≥{min_size}) | filtered q={chosen_q:.3f}",
            alpha_linewidth=1.8,
            alpha_opacity=0.6,
        )

        # --- F) Coherence before vs after scatter (per clonotype)
        pairs = coherence_before_after_table(
            df_before=df_s,
            df_after=df_after,
            ct_col=ct_col,
            markers=markers,
            min_size=min_size,
            debug_probs=debug_probs,
        )
        coh_csv = s_out / f"coherence_before_vs_after_q{chosen_q:.3f}.csv"
        coh_png = s_out / f"coherence_before_vs_after_q{chosen_q:.3f}.png"
        pairs.to_csv(coh_csv, index=False)
        
        # Get mean coherence change at chosen_q from sweep_df
        chosen_row = sweep_df[np.isclose(sweep_df["percentile"], chosen_q)]
        if not chosen_row.empty:
            mean_coh_change = float(chosen_row["mean_coherence_change"].iloc[0])
        else:
            mean_coh_change = None
        
        plot_coherence_before_after_scatter(
            pairs,
            out_png=coh_png,
            title=f"{dataset_name} | {sample} | coherence before vs after (q={chosen_q:.3f})",
            mean_coherence_change=mean_coh_change,
        )

        # --- G) INDEX row
        index_rows.append({
            "dataset": dataset_name,
            "sample": sample,
            "n_cells_before": int(len(df_s)),
            "n_cells_after": int(len(df_after)),
            "n_clonotypes_ge5_before": int(len(n_b)),
            "n_clonotypes_ge5_after": int(len(n_a)),
            "chosen_q": float(chosen_q),
            "Ts": float(Ts),
            "sweep_csv": str(sweep_csv),
            "sweep_metrics_png": str(sweep_metrics_png),
            "before_after_pdf": str(pdf_out),
            "renyi_base_csv": str(renyi_base_csv),
            "renyi_after_csv": str(renyi_after_csv),
            "coherence_csv": str(coh_csv),
            "coherence_png": str(coh_png),
        })

    # dataset-level: 2x3 overlay alpha entropy sweep (first 6 samples)
    if sweep_entropy_all_samples:
        sweep_all = pd.concat(sweep_entropy_all_samples, ignore_index=True)
        plot_entropy_sweep_overlay_alpha_by_sample_2x3(
            sweep_all,
            out_png=ds_out / "ly49c_entropy_sweep_overlay_alpha0to5_2x3.png",
            title=f"{dataset_name} | ΔHα(q) sweep overlay α=0..5 (first 6 samples)",
            chosen_q=chosen_q
        )

    index_df = pd.DataFrame(index_rows)
    index_df.to_csv(ds_out / "INDEX.csv", index=False)
    return index_df

In [43]:
df_b10br = pd.read_csv(
    "../Data/20260116 Comparison 3/20251223 BL6-B10BR HTx HIL Clonotypes with ADT Counts.csv",
    index_col=0
)

df_balbc = pd.read_csv(
    "../Data/20260116 Comparison 3/20251218 BL6-BALBc HTx HIL Clonotypes with ADT Counts.csv",
    index_col=0
)

In [44]:

out_b10br = run_samplewise_pipeline(
    df=df_b10br,
    dataset_name="B10BR_HIL",
    markers=markers_b10br,
    peptides=peptides_b10br,
    output_root=Path("Comparison3_Samplewise_Outputs"),
    thresholds=thresholds,
    chosen_q=0.925,
    min_size=5,
    entropy_alpha_for_sweep=1.0,
    renyi_orders=[0, 1, 2, 3, 4, 5],
    debug_probs=False
)

out_balbc = run_samplewise_pipeline(
    df=df_balbc,
    dataset_name="BALBc_HIL",
    markers=markers_balbc,
    peptides=peptides_balbc,
    output_root=Path("Comparison3_Samplewise_Outputs"),
    thresholds=thresholds,
    chosen_q=0.925,
    min_size=5,
    entropy_alpha_for_sweep=1.0,
    renyi_orders=[0, 1, 2, 3, 4, 5],
    debug_probs=False
)